# Run Other Models Using Database with Payercode

The database with the missing `payer_code` turned to "Unknown" performed overall better than the other two models. Let's run the database against the other models to compare which model produces better predictive outcomes

## Forming df_with_payer database for Model Exploration

In [2]:
# installing libraries
%pip install seaborn
%pip install missingno
%pip install xgboost
%pip install catboost
%pip install regex
%pip install sklearn
%pip install pandas
%pip install numpy
%pip install imblearn
%pip install lightgbm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies ... done
  Getting requireme

In [16]:
# importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, accuracy_score,
                              precision_score, recall_score, roc_curve)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer

In [4]:
# Load + Clean Data
diabetes_original = pd.read_csv('UCI_diabetes/diabetic_data.csv')

diabetes_original.replace('?', np.nan, inplace=True)

diabetes_original.drop(columns=['weight', 'encounter_id', 'patient_nbr',
                                 'max_glu_serum', 'A1Cresult'], inplace=True)

invalid_admission_type        = {5, 6, 8}
invalid_discharge_disposition = {18, 25, 26}
invalid_admission_source      = {9, 15, 17, 20, 21}

diabetes_original['admission_type_id'] = diabetes_original['admission_type_id'].apply(
    lambda x: 'Unknown' if x in invalid_admission_type else str(x))
diabetes_original['discharge_disposition_id'] = diabetes_original['discharge_disposition_id'].apply(
    lambda x: 'Unknown' if x in invalid_discharge_disposition else str(x))
diabetes_original['admission_source_id'] = diabetes_original['admission_source_id'].apply(
    lambda x: 'Unknown' if x in invalid_admission_source else str(x))

cols_to_replace = ['medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'race']
diabetes_original[cols_to_replace] = diabetes_original[cols_to_replace].replace(
    r'^\s*\?\s*$', np.nan, regex=True)

cols_fill_unknown = ['medical_specialty', 'diag_1', 'diag_2', 'diag_3']
diabetes_original[cols_fill_unknown] = diabetes_original[cols_fill_unknown].fillna('Unknown')

def classify_icd9(code_str):
    if pd.isna(code_str) or code_str == 'Unknown':
        return 'Unknown'
    code_str = str(code_str).strip()
    if code_str.startswith('V'): return 'Supplementary'
    if code_str.startswith('E'): return 'External_Causes'
    try:
        code_num = float(code_str)
    except ValueError:
        return 'Unknown'
    if 1 <= code_num < 140:   return 'Infectious_Parasitic'
    elif 140 <= code_num < 240: return 'Neoplasms'
    elif 240 <= code_num < 280: return 'Endocrine'
    elif 280 <= code_num < 290: return 'Blood'
    elif 290 <= code_num < 320: return 'Mental'
    elif 320 <= code_num < 390: return 'Nervous_System'
    elif 390 <= code_num < 460: return 'Circulatory'
    elif 460 <= code_num < 520: return 'Respiratory'
    elif 520 <= code_num < 580: return 'Digestive'
    elif 580 <= code_num < 630: return 'Genitourinary'
    elif 630 <= code_num < 680: return 'Pregnancy_Childbirth'
    elif 680 <= code_num < 710: return 'Skin'
    elif 710 <= code_num < 740: return 'Musculoskeletal'
    elif 740 <= code_num < 760: return 'Congenital'
    elif 760 <= code_num < 780: return 'Perinatal'
    elif 780 <= code_num < 800: return 'Symptoms_Signs'
    elif 800 <= code_num < 1000: return 'Injury_Poisoning'
    else: return 'Unknown'

for col in ['diag_1', 'diag_2', 'diag_3']:
    diabetes_original[col] = diabetes_original[col].apply(classify_icd9)

age_mapping = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3,
    '[40-50)': 4, '[50-60)': 5, '[60-70)': 6, '[70-80)': 7,
    '[80-90)': 8, '[90-100)': 9
}
diabetes_original['age'] = diabetes_original['age'].map(age_mapping)

readmit_map = {'<30': 1, '>30': 0, 'NO': 0, 1: 1, 0: 0, '1': 1, '0': 0}
diabetes_original['readmitted'] = diabetes_original['readmitted'].map(readmit_map)

print(f"Base cleaned data shape: {diabetes_original.shape}")

Base cleaned data shape: (101766, 45)


In [5]:
# Build df_with_payer
df_with_payer = diabetes_original.copy()
df_with_payer['payer_code'] = df_with_payer['payer_code'].fillna('Unknown')
df_with_payer = df_with_payer.dropna(subset=['race'])
print(f"df_with_payer shape: {df_with_payer.shape}")


# Split data into train/valid/test
temp_df, test_df = train_test_split(df_with_payer, test_size=0.15,
                                     random_state=42, stratify=df_with_payer['readmitted'])
train_df, valid_df = train_test_split(temp_df, test_size=0.176,
                                       random_state=42, stratify=temp_df['readmitted'])

target_col = 'readmitted'
x_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
x_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]
x_test  = test_df.drop(columns=[target_col])
y_test  = test_df[target_col]

print(f"Train: {x_train.shape} | Valid: {x_valid.shape} | Test: {x_test.shape}")

# Ordinal encode medication columns
med_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3}
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]
for col in med_cols:
    if col in x_train.columns:
        x_train[col] = x_train[col].map(med_map)
        x_valid[col] = x_valid[col].map(med_map)
        x_test[col]  = x_test[col].map(med_map)

# Binary encode change and diabetesMed
x_train['change'] = (x_train['change'] == 'Ch').astype(int)
x_valid['change'] = (x_valid['change'] == 'Ch').astype(int)
x_test['change']  = (x_test['change']  == 'Ch').astype(int)

x_train['diabetesMed'] = (x_train['diabetesMed'] == 'Yes').astype(int)
x_valid['diabetesMed'] = (x_valid['diabetesMed'] == 'Yes').astype(int)
x_test['diabetesMed']  = (x_test['diabetesMed']  == 'Yes').astype(int)

# One-hot encode nominal columns
nominal_cols = ['race', 'gender', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3',
                'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
                'payer_code']

x_train = pd.get_dummies(x_train, columns=nominal_cols, drop_first=True)
x_valid = pd.get_dummies(x_valid, columns=nominal_cols, drop_first=True)
x_test  = pd.get_dummies(x_test,  columns=nominal_cols, drop_first=True)

x_valid = x_valid.reindex(columns=x_train.columns, fill_value=0)
x_test  = x_test.reindex(columns=x_train.columns,  fill_value=0)

print(f"Encoded shape: {x_train.shape}")

# scale continuous features and apply SMOTE
continuous_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                   'num_medications', 'number_outpatient', 'number_emergency',
                   'number_inpatient', 'number_diagnoses', 'age']

scaler = StandardScaler()
x_train[continuous_cols] = scaler.fit_transform(x_train[continuous_cols])
x_valid[continuous_cols] = scaler.transform(x_valid[continuous_cols])
x_test[continuous_cols]  = scaler.transform(x_test[continuous_cols])


smote = SMOTE(random_state=42, k_neighbors=5)
x_train_res, y_train_res = smote.fit_resample(x_train, y_train)

print("Y counts after SMOTE:")
print(pd.Series(y_train_res).value_counts())
print("\nY ratio after SMOTE:")
print(pd.Series(y_train_res).value_counts(normalize=True))

df_with_payer shape: (99493, 45)
Train: (69684, 44) | Valid: (14885, 44) | Test: (14924, 44)
Encoded shape: (69684, 220)
Y counts after SMOTE:
readmitted
0    61861
1    61861
Name: count, dtype: int64

Y ratio after SMOTE:
readmitted
0    0.5
1    0.5
Name: proportion, dtype: float64


In [6]:
# Helper Functions 

def calc_specificity(y_actual, y_pred, thresh):
    return sum((y_pred < thresh) & (y_actual == 0)) / sum(y_actual == 0)

def calc_prevalence(y_actual):
    return (sum(y_actual) / len(y_actual))

def print_report(y_actual, y_pred, thresh=0.5):
    auc       = roc_auc_score(y_actual, y_pred)
    accuracy  = accuracy_score(y_actual, (y_pred > thresh))
    recall    = recall_score(y_actual, (y_pred > thresh))
    precision = precision_score(y_actual, (y_pred > thresh))
    specificity = calc_specificity(y_actual, y_pred, thresh)
    print('AUC:%.3f'         % auc)
    print('Accuracy:%.3f'    % accuracy)
    print('recall:%.3f'      % recall)
    print('precision:%.3f'   % precision)
    print('specificity:%.3f' % specificity)
    print('prevalence:%.3f'  % calc_prevalence(y_actual))
    print(' ')
    return auc, accuracy, recall, precision, specificity


# Model Comparisons

### Logistic Regression

In [7]:
lr = LogisticRegression(random_state=42, solver='newton-cg', max_iter=1000)
lr.fit(x_train_res, y_train_res)

lr_preds = lr.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, lr_preds)
lr_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {lr_best_thresh:.3f}")
print('Metrics for Validation data — Logistic Regression:')

lr_valid_auc, lr_valid_accuracy, lr_valid_recall, \
    lr_valid_precision, lr_valid_specificity = print_report(y_valid, lr_preds, lr_best_thresh)

Optimal threshold: 0.213
Metrics for Validation data — Logistic Regression:
AUC:0.610
Accuracy:0.709
recall:0.433
precision:0.176
specificity:0.744
prevalence:0.112
 


### KNN Model

In [8]:
knn = KNeighborsClassifier(n_neighbors=100, n_jobs=-1)
knn.fit(x_train_res, y_train_res)

knn_preds = knn.predict_proba(x_valid)[:,1]

fpr, tpr, thresholds = roc_curve(y_valid, knn_preds)
knn_best_thresh = thresholds[np.argmax(tpr-fpr)]
print(f"Optimal threshold: {knn_best_thresh:.3f}")
print('Metrics for Validation data — KNN:')

knn_valid_auc, knn_valid_accuracy, knn_valid_recall, \
    knn_valid_precision, knn_valid_specificity = print_report(y_valid, knn_preds, knn_best_thresh)


Optimal threshold: 0.310
Metrics for Validation data — KNN:
AUC:0.615
Accuracy:0.495
recall:0.679
precision:0.140
specificity:0.449
prevalence:0.112
 


### Linear SVC

In [ ]:
lsvc_clf = LinearSVC(random_state=42, max_iter=1000)
lsvc_clf.fit(x_train_res, y_train_res)

lsvc_preds_raw = lsvc_clf.decision_function(x_valid)

# Normalize decision function scores to [0,1] for threshold + roc_curve
lsvc_scaler = MinMaxScaler()
lsvc_preds = lsvc_scaler.fit_transform(lsvc_preds_raw.reshape(-1, 1)).flatten()

fpr, tpr, thresholds = roc_curve(y_valid, lsvc_preds)
lsvc_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {lsvc_best_thresh:.3f}")
print('Metrics for Validation data — Linear SVC:')

lsvc_valid_auc, lsvc_valid_accuracy, lsvc_valid_recall, \
    lsvc_valid_precision, lsvc_valid_specificity = print_report(y_valid, lsvc_preds, lsvc_best_thresh)

Optimal threshold: 0.762
Metrics for Validation data — Linear SVC:
AUC:0.614
Accuracy:0.716
recall:0.431
precision:0.180
specificity:0.752
prevalence:0.112
 


### Decision Tree

In [10]:
dc_clf = DecisionTreeClassifier(random_state=42, max_depth=10)
dc_clf.fit(x_train_res, y_train_res)

dc_preds = dc_clf.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, dc_preds)
dc_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {dc_best_thresh:.3f}")
print('Metrics for Validation data — Decision Tree:')

dc_valid_auc, dc_valid_accuracy, dc_valid_recall, \
    dc_valid_precision, dc_valid_specificity = print_report(y_valid, dc_preds, dc_best_thresh)

Optimal threshold: 0.335
Metrics for Validation data — Decision Tree:
AUC:0.585
Accuracy:0.659
recall:0.450
precision:0.153
specificity:0.661
prevalence:0.112
 


### Stochastic Gradient Descent Model

In [11]:
sgdc = SGDClassifier(loss='log_loss', alpha=0.1, random_state=42)
sgdc.fit(x_train_res, y_train_res)

sgd_preds = sgdc.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, sgd_preds)
sgd_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {sgd_best_thresh:.3f}")
print('Metrics for Validation data — SGD Classifier:')

sgdc_valid_auc, sgdc_valid_accuracy, sgdc_valid_recall, \
    sgdc_valid_precision, sgdc_valid_specificity = print_report(y_valid, sgd_preds, sgd_best_thresh)

Optimal threshold: 0.463
Metrics for Validation data — SGD Classifier:
AUC:0.620
Accuracy:0.606
recall:0.561
precision:0.155
specificity:0.612
prevalence:0.112
 


### Random Forest

In [12]:
rf_clf = RandomForestClassifier(random_state=111, max_depth=6, n_estimators=300, n_jobs=-1)
rf_clf.fit(x_train_res, y_train_res)

rf_preds = rf_clf.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, rf_preds)
rf_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {rf_best_thresh:.3f}")
print('Metrics for Validation data — Random Forest:')

rf_valid_auc, rf_valid_accuracy, rf_valid_recall, \
    rf_valid_precision, rf_valid_specificity = print_report(y_valid, rf_preds, rf_best_thresh)

Optimal threshold: 0.475
Metrics for Validation data — Random Forest:
AUC:0.623
Accuracy:0.645
recall:0.524
precision:0.163
specificity:0.661
prevalence:0.112
 


### Gradient Boosting

In [13]:
gb_clf = GradientBoostingClassifier(n_estimators=100, criterion='friedman_mse',
                                     learning_rate=1.0, max_depth=3, random_state=42)
gb_clf.fit(x_train_res, y_train_res)

gb_preds = gb_clf.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, gb_preds)
gb_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {gb_best_thresh:.3f}")
print('Metrics for Validation data — Gradient Boosting:')

gb_valid_auc, gb_valid_accuracy, gb_valid_recall, \
    gb_valid_precision, gb_valid_specificity = print_report(y_valid, gb_preds, gb_best_thresh)

Optimal threshold: 0.120
Metrics for Validation data — Gradient Boosting:
AUC:0.647
Accuracy:0.660
recall:0.536
precision:0.173
specificity:0.675
prevalence:0.112
 


### XGBoost

In [14]:
xgb_clf = xgb.XGBClassifier(max_depth=3, learning_rate=1.0,
                              eval_metric='logloss', random_state=42)
xgb_clf.fit(x_train_res, y_train_res)

xgb_preds = xgb_clf.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, xgb_preds)
xgb_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {xgb_best_thresh:.3f}")
print('Metrics for Validation data — XGBoost:')

xgb_valid_auc, xgb_valid_accuracy, xgb_valid_recall, \
    xgb_valid_precision, xgb_valid_specificity = print_report(y_valid, xgb_preds, xgb_best_thresh)

Optimal threshold: 0.131
Metrics for Validation data — XGBoost:
AUC:0.649
Accuracy:0.676
recall:0.518
precision:0.177
specificity:0.696
prevalence:0.112
 


### CatBoost

In [15]:
catb = CatBoostClassifier(iterations=200, depth=3, learning_rate=1.0,
                           random_state=111, verbose=0)
catb.fit(x_train_res, y_train_res)

catb_preds = catb.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, catb_preds)
catb_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {catb_best_thresh:.3f}")
print('Metrics for Validation data — CatBoost:')

catb_valid_auc, catb_valid_accuracy, catb_valid_recall, \
    catb_valid_precision, catb_valid_specificity = print_report(y_valid, catb_preds, catb_best_thresh)

Optimal threshold: 0.117
Metrics for Validation data — CatBoost:
AUC:0.655
Accuracy:0.646
recall:0.567
precision:0.172
specificity:0.656
prevalence:0.112
 


# Hyper Parameter Tuning

## Decision Tree - Hyper Parameter Tuning

In [17]:
recall_scoring = make_scorer(recall_score)

dc_grid = {
    'max_features':      ['sqrt', 'log2'],
    'max_depth':         range(1, 11, 1),
    'min_samples_split': range(2, 10, 2),
    'criterion':         ['gini', 'entropy']
}

dc_random = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_distributions=dc_grid,
    n_iter=20, cv=2, scoring=recall_scoring,
    verbose=1, random_state=111
)
dc_random.fit(x_train_res, y_train_res)

print(dc_random.best_params_)

dc_hp_preds       = dc_random.best_estimator_.predict(x_valid)
dc_hp_preds_proba = dc_random.best_estimator_.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, dc_hp_preds_proba)
dc_hp_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {dc_hp_best_thresh:.3f}")

dc_hp_auc, dc_hp_accuracy, dc_hp_recall, \
    dc_hp_precision, dc_hp_specificity = print_report(y_valid, dc_hp_preds_proba, dc_hp_best_thresh)

Fitting 2 folds for each of 20 candidates, totalling 40 fits
{'min_samples_split': 2, 'max_features': 'log2', 'max_depth': 7, 'criterion': 'entropy'}
Optimal threshold: 0.520
AUC:0.533
Accuracy:0.523
recall:0.540
precision:0.125
specificity:0.520
prevalence:0.112
 


## Random Forest - Hypertuning

In [19]:
rf_grid = {
    'n_estimators':      range(200, 1000, 200),
    'max_features':      ['sqrt', 'log2'],
    'max_depth':         range(1, 11, 1),
    'min_samples_split': range(2, 10, 2),
    'criterion':         ['gini', 'entropy']
}

rf_random = RandomizedSearchCV(
    RandomForestClassifier(random_state=111, n_jobs=-1),
    param_distributions=rf_grid,
    n_iter=20, cv=2, scoring=recall_scoring,
    verbose=1, random_state=111
)
rf_random.fit(x_train_res, y_train_res)

print(rf_random.best_params_)

rf_hp_preds       = rf_random.best_estimator_.predict(x_valid)
rf_hp_preds_proba = rf_random.best_estimator_.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, rf_hp_preds_proba)
rf_hp_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {rf_hp_best_thresh:.3f}")

rf_hp_auc, rf_hp_accuracy, rf_hp_recall, \
    rf_hp_precision, rf_hp_specificity = print_report(y_valid, rf_hp_preds_proba, rf_hp_best_thresh)

Fitting 2 folds for each of 20 candidates, totalling 40 fits
{'n_estimators': 800, 'min_samples_split': 2, 'max_features': 'log2', 'max_depth': 10, 'criterion': 'gini'}
Optimal threshold: 0.467
AUC:0.628
Accuracy:0.660
recall:0.507
precision:0.167
specificity:0.679
prevalence:0.112
 


## XGBoost - Hypertuning

In [20]:
xgb_grid = {
    'min_child_weight': [1, 5, 8, 10],
    'gamma':            [0.5, 1, 1.5, 2, 5],
    'subsample':        [0.6, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.8, 0.9, 1.0],
    'max_depth':        [3, 4, 5]
}

xgb_random = GridSearchCV(
    xgb.XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=xgb_grid,
    cv=2, scoring=recall_scoring, verbose=1
)
xgb_random.fit(x_train_res, y_train_res)

print(xgb_random.best_params_)

xgb_hp_preds       = xgb_random.best_estimator_.predict(x_valid)
xgb_hp_preds_proba = xgb_random.best_estimator_.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, xgb_hp_preds_proba)
xgb_hp_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {xgb_hp_best_thresh:.3f}")

xgb_hp_auc, xgb_hp_accuracy, xgb_hp_recall, \
    xgb_hp_precision, xgb_hp_specificity = print_report(y_valid, xgb_hp_preds_proba, xgb_hp_best_thresh)

Fitting 2 folds for each of 960 candidates, totalling 1920 fits
{'colsample_bytree': 1.0, 'gamma': 1.5, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.9}
Optimal threshold: 0.142
AUC:0.662
Accuracy:0.665
recall:0.554
precision:0.179
specificity:0.679
prevalence:0.112
 


## CatBoost - Hypertuning

In [21]:
catb_grid = {
    'iterations':    [100, 200, 300],
    'depth':         [3, 4, 5],
    'learning_rate': [0.01, 0.1, 1.0]
}

catb_random = GridSearchCV(
    CatBoostClassifier(random_state=111, verbose=0),
    param_grid=catb_grid,
    cv=2, scoring=recall_scoring, verbose=1
)
catb_random.fit(x_train_res, y_train_res)

print(catb_random.best_params_)

catb_hp_preds       = catb_random.best_estimator_.predict(x_valid)
catb_hp_preds_proba = catb_random.best_estimator_.predict_proba(x_valid)[:, 1]

fpr, tpr, thresholds = roc_curve(y_valid, catb_hp_preds_proba)
catb_hp_best_thresh = thresholds[np.argmax(tpr - fpr)]
print(f"Optimal threshold: {catb_hp_best_thresh:.3f}")

catb_hp_auc, catb_hp_accuracy, catb_hp_recall, \
    catb_hp_precision, catb_hp_specificity = print_report(y_valid, catb_hp_preds_proba, catb_hp_best_thresh)

Fitting 2 folds for each of 27 candidates, totalling 54 fits
{'depth': 5, 'iterations': 300, 'learning_rate': 0.1}
Optimal threshold: 0.156
AUC:0.675
Accuracy:0.736
recall:0.484
precision:0.208
specificity:0.767
prevalence:0.112
 


## Final Comparison:

In [24]:
results_table = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'KNN',
        'Decision Tree',
        'Random Forest',
        'Gradient Boosting',
        'XGBoost',
        'CatBoost',
        'SGD Classifier',
        'Linear SVC',
        'Decision Tree (Tuned)',
        'Random Forest (Tuned)',
        'XGBoost (Tuned)',
        'CatBoost (Tuned)'
    ],
    'AUC': [
        lr_valid_auc, knn_valid_auc, dc_valid_auc, rf_valid_auc,
        gb_valid_auc, xgb_valid_auc, catb_valid_auc,
        sgdc_valid_auc, lsvc_valid_auc,
        dc_hp_auc, rf_hp_auc, xgb_hp_auc, catb_hp_auc
    ],
    'Accuracy': [
        lr_valid_accuracy, knn_valid_accuracy, dc_valid_accuracy, rf_valid_accuracy,
        gb_valid_accuracy, xgb_valid_accuracy, catb_valid_accuracy,
        sgdc_valid_accuracy, lsvc_valid_accuracy,
        dc_hp_accuracy, rf_hp_accuracy, xgb_hp_accuracy, catb_hp_accuracy
    ],
    'Recall': [
        lr_valid_recall, knn_valid_recall, dc_valid_recall, rf_valid_recall,
        gb_valid_recall, xgb_valid_recall, catb_valid_recall,
        sgdc_valid_recall, lsvc_valid_recall,
        dc_hp_recall, rf_hp_recall, xgb_hp_recall, catb_hp_recall
    ],
    'Precision': [
        lr_valid_precision, knn_valid_precision, dc_valid_precision, rf_valid_precision,
        gb_valid_precision, xgb_valid_precision, catb_valid_precision,
        sgdc_valid_precision, lsvc_valid_precision,
        dc_hp_precision, rf_hp_precision, xgb_hp_precision, catb_hp_precision
    ],
    'Specificity': [
        lr_valid_specificity, knn_valid_specificity, dc_valid_specificity, rf_valid_specificity,
        gb_valid_specificity, xgb_valid_specificity, catb_valid_specificity,
        sgdc_valid_specificity, lsvc_valid_specificity,
        dc_hp_specificity, rf_hp_specificity, xgb_hp_specificity, catb_hp_specificity
    ],
    'Prevalence': [calc_prevalence(y_valid)] * 13
})

results_table = results_table.sort_values('AUC', ascending=False).reset_index(drop=True)

print('=' * 90)
print('  MODEL COMPARISON — WITH PAYER_CODE CONFIGURATION')
print('=' * 90)
print(results_table.to_string(index=False))


  MODEL COMPARISON — WITH PAYER_CODE CONFIGURATION
                Model      AUC  Accuracy   Recall  Precision  Specificity  Prevalence
     CatBoost (Tuned) 0.674896  0.735506 0.484141   0.208290     0.767292    0.112261
      XGBoost (Tuned) 0.661634  0.664830 0.554159   0.179110     0.678825    0.112261
             CatBoost 0.654712  0.645818 0.566727   0.172338     0.655820    0.112261
              XGBoost 0.648684  0.676318 0.517654   0.177363     0.696383    0.112261
    Gradient Boosting 0.647152  0.659792 0.536206   0.172806     0.675420    0.112261
Random Forest (Tuned) 0.627599  0.659859 0.507481   0.166667     0.679128    0.112261
        Random Forest 0.622590  0.645415 0.524237   0.163463     0.660739    0.112261
       SGD Classifier 0.620186  0.606449 0.561341   0.154709     0.612154    0.112261
                  KNN 0.614703  0.495129 0.678636   0.139793     0.449372    0.112261
           Linear SVC 0.613749  0.715620 0.430880   0.179910     0.751627    0.112261
  L

#### Model Selection: CatBoost (Tuned)

After comparing 13 models across AUC, Recall, Precision, Accuracy, and Specificity, 
**CatBoost (Tuned)** was selected as the final model based on its overall performance across all metrics.

CatBoost (Tuned) is the most well-rounded model in the comparison. While other models may 
outperform it on a single metric, no other model consistently performs at or near the top 
across all five metrics simultaneously.

**CatBoost (Tuned) leads the comparison in 4 out of 5 key metrics:**
- **AUC (0.675)** — Best overall ability to distinguish between readmitted and non-readmitted patients
- **Accuracy (0.736)** — Highest overall correctness across all predictions
- **Precision (0.208)** — When it flags a patient as high risk, it is right more often than any other model
- **Specificity (0.767)** — Least likely to incorrectly flag patients who will not be readmitted, reducing wasted clinical resources

#### What About KNN's Higher Recall?

KNN has the highest recall at 0.679 compared to CatBoost's 0.484. However, KNN achieves 
this by flagging nearly every patient as high risk, which is reflected in its poor Specificity 
(0.449) and overall Accuracy (0.495) — barely better than random chance. In a clinical setting, 
this level of false alarms would quickly erode staff trust in the model.

XGBoost (Tuned) is the closest competitor with a higher Recall (0.554 vs 0.484). However, 
CatBoost outperforms it on AUC, Accuracy, Precision, and Specificity — winning 4 out of 5 
key metrics.

#### Conclusion

CatBoost (Tuned) provides the best balance between catching true readmissions and minimizing 
false alarms. For a clinical readmission model where both accuracy and trust matter, 
CatBoost (Tuned) is the most reliable and actionable choice.